In [4]:
text = "the cat in the hat. Maayon and seyon are very good boys."
tokens = list(text.encode('utf-8'))
print('no of tokens: ', len(tokens))
print(tokens)

no of tokens:  56
[116, 104, 101, 32, 99, 97, 116, 32, 105, 110, 32, 116, 104, 101, 32, 104, 97, 116, 46, 32, 77, 97, 97, 121, 111, 110, 32, 97, 110, 100, 32, 115, 101, 121, 111, 110, 32, 97, 114, 101, 32, 118, 101, 114, 121, 32, 103, 111, 111, 100, 32, 98, 111, 121, 115, 46]


In [32]:
def get_pair_counts(token_ids):
    """Count frequency of each adjacent pair in the token sequence."""
    counts = {}
    for i in range(len(token_ids) - 1):
        pair = (token_ids[i], token_ids[i+1])
        counts[pair] = counts.get(pair, 0) + 1
    return counts

def merge(token_ids, pair, new_id):
    """Replace all occurrences of pair in token_ids with new_id."""
    new_tokens = []
    i = 0
    while i < len(token_ids):
        # If we found the pair, merge it
        if i < len(token_ids) - 1 and (token_ids[i], token_ids[i + 1]) == pair:
            new_tokens.append(new_id)
            i += 2  # Skip both tokens in the pair
        else:
            new_tokens.append(token_ids[i])
            i += 1
    return new_tokens

def train(text, vocab_size):
    """Train BPE on text, returning merge rules and vocabulary."""
    assert vocab_size >= 256, "Vocab size must be at least 256"
    num_merges = vocab_size - 256

    tokens = list(text.encode("utf-8"))
    merges = {}
    vocab = {i: bytes([i]) for i in range(256)}

    for i in range(num_merges):
        pair_counts = get_pair_counts(tokens)
        if not pair_counts:
            break

        best_pair = max(pair_counts, key=pair_counts.get)
        new_id = 256 + i
        tokens = merge(tokens, best_pair, new_id)
        merges[best_pair] = new_id
        vocab[new_id] = vocab[best_pair[0]] + vocab[best_pair[1]]

        print(f"Merge {i+1}: {best_pair} -> {new_id} "
              f"({vocab[new_id]}) | Tokens: {len(tokens)}")

    return merges, vocab

def encode(text, merges):
    """Encode text into token IDs using learned merge rules."""
    tokens = list(text.encode("utf-8"))
    while len(tokens) >= 2:
        pair_counts = get_pair_counts(tokens)
        candidate = None
        for pair in pair_counts:
            if pair in merges:
                if candidate is None or merges[pair] < merges[candidate]:
                    candidate = pair
        if candidate is None:
            break
        tokens = merge(tokens, candidate, merges[candidate])
    return tokens

def decode(token_ids, vocab):
    """Decode token IDs back to text."""
    byte_sequence = b"".join(vocab[tid] for tid in token_ids)
    return byte_sequence.decode("utf-8", errors="replace")



# test functions

In [11]:
tokens = list(text.encode('utf-8'))
print('no of tokens: ', len(tokens))
print(tokens)
pair_counts = get_pair_counts(token_ids=tokens)
print(pair_counts)

# Show top 5 most frequent pairs
top_pairs = sorted(pair_counts.items(), key=lambda x: x[1], reverse=True)[:5]
for pair, count in top_pairs:
    print(f"({chr(pair[0])}, {chr(pair[1])}) -> count: {count}")



no of tokens:  56
[116, 104, 101, 32, 99, 97, 116, 32, 105, 110, 32, 116, 104, 101, 32, 104, 97, 116, 46, 32, 77, 97, 97, 121, 111, 110, 32, 97, 110, 100, 32, 115, 101, 121, 111, 110, 32, 97, 114, 101, 32, 118, 101, 114, 121, 32, 103, 111, 111, 100, 32, 98, 111, 121, 115, 46]
{(116, 104): 2, (104, 101): 2, (101, 32): 3, (32, 99): 1, (99, 97): 1, (97, 116): 2, (116, 32): 1, (32, 105): 1, (105, 110): 1, (110, 32): 3, (32, 116): 1, (32, 104): 1, (104, 97): 1, (116, 46): 1, (46, 32): 1, (32, 77): 1, (77, 97): 1, (97, 97): 1, (97, 121): 1, (121, 111): 2, (111, 110): 2, (32, 97): 2, (97, 110): 1, (110, 100): 1, (100, 32): 2, (32, 115): 1, (115, 101): 1, (101, 121): 1, (97, 114): 1, (114, 101): 1, (32, 118): 1, (118, 101): 1, (101, 114): 1, (114, 121): 1, (121, 32): 1, (32, 103): 1, (103, 111): 1, (111, 111): 1, (111, 100): 1, (32, 98): 1, (98, 111): 1, (111, 121): 1, (121, 115): 1, (115, 46): 1}
(e,  ) -> count: 3
(n,  ) -> count: 3
(t, h) -> count: 2
(h, e) -> count: 2
(a, t) -> count: 2


In [ ]:
merged = merge(tokens, (116, 104), 256)
print(f"Before:  {len(tokens)} tokens")
print(f"After:  {len(merged)} tokens")
print(f"Merged sequence: {merged}")

Before:  56 tokens
After:  54 tokens
Merged sequence: [256, 101, 32, 99, 97, 116, 32, 105, 110, 32, 256, 101, 32, 104, 97, 116, 46, 32, 77, 97, 97, 121, 111, 110, 32, 97, 110, 100, 32, 115, 101, 121, 111, 110, 32, 97, 114, 101, 32, 118, 101, 114, 121, 32, 103, 111, 111, 100, 32, 98, 111, 121, 115, 46]


# Train the tokenizer

In [28]:
# Train on sample text with vocab_size = 266 (10 merges) ie. 256-266 = 10
training_text = text + "the cat in the hat sat on the mat. the cat and the hat."
merges, vocab = train(training_text, vocab_size=266)
print(f"Before:  {len(training_text)} tokens")
print(f"After:  {len(merges)} tokens")
print(f"Merged sequence: {merges}")
print(f"Vocabulary after 256 char: {dict(list(vocab.items())[256:])}")

Merge 1: (101, 32) -> 256 (b'e ') | Tokens: 103
Merge 2: (97, 116) -> 257 (b'at') | Tokens: 95
Merge 3: (116, 104) -> 258 (b'th') | Tokens: 88
Merge 4: (258, 256) -> 259 (b'the ') | Tokens: 81
Merge 5: (257, 32) -> 260 (b'at ') | Tokens: 76
Merge 6: (110, 32) -> 261 (b'n ') | Tokens: 71
Merge 7: (259, 99) -> 262 (b'the c') | Tokens: 68
Merge 8: (262, 260) -> 263 (b'the cat ') | Tokens: 65
Merge 9: (261, 259) -> 264 (b'n the ') | Tokens: 62
Merge 10: (257, 46) -> 265 (b'at.') | Tokens: 59
Before:  111 tokens
After:  10 tokens
Merged sequence: {(101, 32): 256, (97, 116): 257, (116, 104): 258, (258, 256): 259, (257, 32): 260, (110, 32): 261, (259, 99): 262, (262, 260): 263, (261, 259): 264, (257, 46): 265}
Vocabulary after 256 char: {256: b'e ', 257: b'at', 258: b'th', 259: b'the ', 260: b'at ', 261: b'n ', 262: b'the c', 263: b'the cat ', 264: b'n the ', 265: b'at.'}


## Why order matters in BPE

BPE learns a hierarchy of merges. Early merges capture common character pairs like “th”. Later merges build on earlier ones to form full words like “the” and even phrases like “. the “. This hierarchy is why merge order matters during encoding.

# Encode and Decode

Note: The encode-decode round trip preserves the original text perfectly. This is a fundamental requirement — a tokenizer that loses information during encoding is useless.

In [31]:
encoded = encode("the cat in the hat", merges)
print(f"Encoded: {encoded}")
print(f"Number of tokens: {len(encoded)}")

Encoded: [263, 105, 264, 104, 257]
Number of tokens: 5


In [33]:
decoded = decode(encoded, vocab)
print(f"Decoded: '{decoded}'")
print(f"Round-trip matches: {decoded == 'the cat in the hat'}")

Decoded: 'the cat in the hat'
Round-trip matches: True


# Exercise 1: Experiment with Vocabulary Size

Train the BPE tokenizer on the text below with two different vocabulary sizes: 260 and 280. Compare how many tokens each produces when encoding the test sentence. Which vocab size compresses better?

A larger vocabulary captures longer patterns, so it produces fewer tokens. But the gains diminish — the biggest compression comes from the first merges. In production, models choose vocab sizes that balance compression against embedding table memory.

In [34]:
corpus = "the cat in the hat sat on the mat. the cat and the hat. the cat is a fat cat."
test_sentence = "the fat cat sat on the mat"

merges_260, vocab_260 = train(corpus, 260)
merges_280, vocab_280 = train(corpus, 280)

tokens_260 = encode(test_sentence, merges_260)
tokens_280 = encode(test_sentence, merges_280)

print(f"Vocab 260: {len(tokens_260)} tokens → {tokens_260}")
print(f"Vocab 280: {len(tokens_280)} tokens → {tokens_280}")
print(f"Compression improvement: {len(tokens_260) - len(tokens_280)} fewer tokens")

Merge 1: (97, 116) -> 256 (b'at') | Tokens: 68
Merge 2: (116, 104) -> 257 (b'th') | Tokens: 62
Merge 3: (257, 101) -> 258 (b'the') | Tokens: 56
Merge 4: (258, 32) -> 259 (b'the ') | Tokens: 50
Merge 1: (97, 116) -> 256 (b'at') | Tokens: 68
Merge 2: (116, 104) -> 257 (b'th') | Tokens: 62
Merge 3: (257, 101) -> 258 (b'the') | Tokens: 56
Merge 4: (258, 32) -> 259 (b'the ') | Tokens: 50
Merge 5: (256, 32) -> 260 (b'at ') | Tokens: 44
Merge 6: (32, 259) -> 261 (b' the ') | Tokens: 39
Merge 7: (99, 260) -> 262 (b'cat ') | Tokens: 36
Merge 8: (256, 46) -> 263 (b'at.') | Tokens: 33
Merge 9: (262, 105) -> 264 (b'cat i') | Tokens: 31
Merge 10: (110, 261) -> 265 (b'n the ') | Tokens: 29
Merge 11: (263, 261) -> 266 (b'at. the ') | Tokens: 27
Merge 12: (259, 264) -> 267 (b'the cat i') | Tokens: 26
Merge 13: (267, 265) -> 268 (b'the cat in the ') | Tokens: 25
Merge 14: (268, 104) -> 269 (b'the cat in the h') | Tokens: 24
Merge 15: (269, 260) -> 270 (b'the cat in the hat ') | Tokens: 23
Merge 16: (27

# Putting all together - BasicBPETokenizer.py


In [5]:
from BasicBPETokenizer import BasicBPETokenizer

tokenizer = BasicBPETokenizer()
training_corpus = (
    "the cat in the hat sat on the mat. "
    "the cat and the hat. "
    "the cat is a fat cat."
)
tokenizer.train(training_corpus, vocab_size=276)

test = "the cat sat on the mat"
encoded = tokenizer.encode(test)
decoded = tokenizer.decode(encoded)
print(f"\nOriginal:  '{test}'")
print(f"Encoded:   {encoded} ({len(encoded)} tokens)")
print(f"Decoded:   '{decoded}'")
print(f"Match:     {test == decoded}")

Trained: 20 merges, vocab size = 276

Original:  'the cat sat on the mat'
Encoded:   [259, 262, 115, 260, 111, 265, 109, 256] (8 tokens)
Decoded:   'the cat sat on the mat'
Match:     True
